**Razonamiento con Agentes de Lenguaje en LangChain**

Este programa permite utilizar el framework [LangChain](https://python.langchain.com/) de OpenAI para desarrollar aplicaciones simples basadas en LLMs. LangChain permite conectar un LLM a otras fuentes de datos (i.e., bases de datos SQL, buscador de Google, etc), y permite  al  LLM interactuar con su ambiente utilizando [Agentes](https://python.langchain.com/docs/modules/agents).

*LangChain* permite generar cadenas de *razonamiento* paso-a-paso para responder a tareas de alto nivel, descomponiéndolas en tareas más simples.
Para lograr esto, un agente tiene acceso a varias herramientas y determina cuál de ellas debe utilizar dependiendo de la entrada del usuario. En general, existen dos tipos de agentes:

1. **Agentes de Acción**: decide la próxima acción utilizando las salidas de las acciones previas. Estos son adecuados para tareas pequeñas.
2. **Agentes de Planificar-y-Ejecutar**: decide sobre la secuencia completa de acciones y luego las ejecuta todas sin actualizar el plan. Estos son adecuados para tareas complejas que requieren mantener objetivos de largo plazo.

Instalamos algunos paquetes tales como LangChain, openai, y buscadores de google:

In [1]:
!pip install  langchain openai pymysql --upgrade -q
!pip install  google-search-results -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.8/786.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.9 MB/s eta 0:00:00


Importamos algunas librerías de LangChain para uso de agentes de languaje  y ajustamos las variables de ambiente para uso de las respectivas APIs de OpenAI (**OPENAI_API_KEY**) y Google (**SERPAPI_API_KEY**), para las cuales Ud. debe obtener las respectivas claves:

In [3]:
from langchain.agents import load_tools
from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain.llms import OpenAI
import os

In [4]:
# Open AI API-key
from google.colab import files
from IPython.display import clear_output

files.upload() # subir archivo con apikey de openai propio
clear_output() # no muestra contenido del apikey

In [5]:
def get_api_key():
    with open('idsa_openai_key.txt', 'r') as fp: #acá reemplazar x el nombre de tu archivo
        key = fp.read()
    return key

# Enter your OpenAI API key here
OPENAI_API_KEY = get_api_key()

In [6]:
os.environ['OPENAI_API_KEY']  = OPENAI_API_KEY
os.environ["SERPAPI_API_KEY"] = "2e903e74f87729885b46d4f2d7be6949003031f2a1adbd145da65435cf04b3ab"

Inicializamos las bibliotecas de OpenAI para LLMs y algunas herramientas a utilizar tales como el buscador de Google (*serpAPI*) y un LLM (por defecto un modelo pre-entrenado de GPT como "*text-davinci-003*"):

In [8]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.7 MB/s eta 0:00:00


In [10]:
from langchain import OpenAI
from langchain.agents import load_tools
from langchain.agents import initialize_agent
from langchain.agents import AgentType

llm = OpenAI()
tools = load_tools(["serpapi", "llm-math"], llm=llm)

En este ejemplo,  utilizamos un agente  simple de lenguaje SIN memoria (*ZERO_SHOT_REACT_DESCRIPTION*). Es decir, la acción que este  realiza se basa solamente en la acción actual y no en las previas (historial). De este modo, el agente decide qué herramienta utilizar basado exclusivamente en la descripción de la herramienta:


In [11]:
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
agent.run("¿Quién es la esposa del presidente de Croacia y cuál será la edad de ella en 10 años más?")

/tmp/ipython-input-1500639022.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
/tmp/ipython-input-1500639022.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent.run("¿Quién es la esposa del presidente de Croacia y 

"The president's wife, Sanja Musić Milanović, will be 67 years old in 10 years."

Ahora, solicitamos al agente que muestre paso-a-paso lo que realizó (verbose):

In [12]:
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
agent.run("¿Quién es la esposa del presidente de Croacia y cuál será la edad de ella en 10 años más?")



> Entering new AgentExecutor chain...
 I should first search for information about the president of Croatia and his wife.
Action: Search
Action Input: "President of Croatia"
Observation: Zoran Milanović
Thought: Now I should search for information about his wife.
Action: Search
Action Input: "Zoran Milanović's wife"
Observation: ['Sanja Musić Milanović is a Croatian scientist and a professor at the Medical School of the University of Zagreb. She is the wife of Zoran Milanović, the fifth President of Croatia and former Prime Minister.', 'Sanja Musić Milanović type: First Lady of Croatia.', 'Sanja Musić Milanović kgmid: /m/0j6jq5j.', 'Sanja Musić Milanović born: 1969 (age 56 years), Zagreb, Croatia.', 'Sanja Musić Milanović education: School of Medicine, University of Zagreb (2010).', 'Sanja Musić Milanović spouse: Zoran Milanović (m. 1994).', 'Sanja Musić Milanović parents: Ivan Musić, Ana Musić.', 'Sanja Musić Milanović office: First Spouse of Croatia.', 'Sanja Musić Milanović nation

'Sanja Musić Milanović will be 67 years old in 10 years.'

In [15]:
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)
agent.run("¿Qué animales no documentados anteriormente encontraron los científicos de la misión 'Underwater Oases of Mar Del Plata Canyon' del Conicet y del Schmidt Ocean Institute?")



> Entering new AgentExecutor chain...
 I should use a search engine to find information about this mission and the animals they discovered.
Action: Search
Action Input: 'Underwater Oases of Mar Del Plata Canyon' Conicet Schmidt Ocean Institute
Observation: ["This expedition will explore the diversity and distribution of seafloor communities in one of the country's largest deep-sea canyons.", 'Underwater Oases of Mar Del Plata Canyon. 23 July – 10 August 2025 #UnderwaterOases. Two powerful currents converge in the Mar del Plata Submarine Canyon in ...', 'Lush with Life | Underwater Oases of the Mar Del Plata Canyon · Colossal Squid, 1st Live Observation | Searching for New Species in the South Sandwich Islands.', 'Underwater Oases of the Mar Del Plata Canyon. View The Cruise. news-icon. Updates. Explore recent news and updates from around the institute and at sea. Cruise ...', 'Investigadores del CONICET realizan la expedición “Underwater Oases of Mar Del Plata Canyon: Talud Continent

"The scientists of the 'Underwater Oases of Mar Del Plata Canyon' mission discovered numerous new species during their expedition, including deep sea corals, fish, and other marine animals."